# Observatório Dengue × Clima — Maringá
## Visualizações para Artigo / Apresentação

**Período:** 2020–2025 (314 semanas)  
**Fontes:** InfoDengue, Open-Meteo ERA5, Google Earth Engine (MODIS)  
**Modelo:** Random Forest com walk-forward validation  

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

from observatorio_dengue.etl import database
from observatorio_dengue.features.cruzamento import cruzar_dengue_clima
from observatorio_dengue.features.correlacoes import matriz_correlacoes

# ── Estilo ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

# ── Diretório de saída para figuras ──────────────────────────────────────
FIG_DIR = Path("reports/figuras")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figuras serão salvas em: {FIG_DIR.resolve()}")

## 1. Carga dos Dados

In [ ]:
# Carrega as 3 fontes do DuckDB
df_dengue = database.carregar("SELECT * FROM dengue_raw ORDER BY ano_epi, semana_epi")
df_clima = database.carregar("SELECT * FROM clima_semanal ORDER BY ano_epi, semana_epi")
df_satelite = database.carregar("SELECT * FROM satelite_municipio_semanal ORDER BY ano_epi, semana_epi")

# Cria coluna de data aproximada (segunda-feira da semana epi) para eixo temporal
from epiweeks import Week
df_dengue["data"] = df_dengue.apply(
    lambda r: Week(int(r["ano_epi"]), int(r["semana_epi"]), system="iso").startdate(),
    axis=1,
)
df_dengue["data"] = pd.to_datetime(df_dengue["data"])

print(f"Dengue:   {len(df_dengue)} semanas ({df_dengue['ano_epi'].min()}-{df_dengue['ano_epi'].max()})")
print(f"Clima:    {len(df_clima)} semanas")
print(f"Satélite: {len(df_satelite)} semanas")

## 2. Série Temporal — Incidência de Dengue (2020–2025)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(df_dengue["data"], df_dengue["p_inc100k"], alpha=0.3, color="crimson")
ax.plot(df_dengue["data"], df_dengue["p_inc100k"], color="crimson", linewidth=1.2, label="Incidência")

# Destaque de surtos (>100 por 100k)
surto = df_dengue[df_dengue["p_inc100k"] > 100]
if not surto.empty:
    ax.scatter(surto["data"], surto["p_inc100k"], color="darkred", s=15, zorder=5, label="Surto (>100/100k)")

ax.set_xlabel("Data")
ax.set_ylabel("Incidência por 100 mil hab.")
ax.set_title("Dengue em Maringá — Incidência Semanal (2020–2025)")
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
fig.savefig(FIG_DIR / "01_serie_temporal_dengue.png")
plt.show()
print("✅ Figura salva: 01_serie_temporal_dengue.png")

## 3. Heatmap — Correlação (Spearman) por Variável × Lag

In [ ]:
# Recalcula matriz de correlações para clima + satélite
vars_clima = ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
              "precipitation_sum", "relative_humidity_2m_mean"]
vars_sat = ["ndvi", "lst_night_c"]

mat_clima = matriz_correlacoes(
    df_dengue, df_clima, vars_clima=vars_clima,
    lags=range(0, 9), coluna_dengue="p_inc100k", metodos=("spearman",),
)
mat_sat = matriz_correlacoes(
    df_dengue, df_satelite, vars_clima=vars_sat,
    lags=range(0, 9), coluna_dengue="p_inc100k", metodos=("spearman",),
)
matriz = pd.concat([mat_clima, mat_sat], ignore_index=True)

# Pivot para heatmap
pivot = matriz.pivot(index="var_clima", columns="lag", values="r")

# Nomes mais limpos
rename_vars = {
    "temperature_2m_mean": "Temp. Média",
    "temperature_2m_max": "Temp. Máxima",
    "temperature_2m_min": "Temp. Mínima",
    "precipitation_sum": "Precipitação",
    "relative_humidity_2m_mean": "Umid. Relativa",
    "ndvi": "NDVI (satélite)",
    "lst_night_c": "LST Noturna (satélite)",
}
pivot.index = pivot.index.map(lambda x: rename_vars.get(x, x))

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
    linewidths=0.5, ax=ax, vmin=-0.2, vmax=0.6,
    cbar_kws={"label": "Correlação de Spearman (ρ)"},
)
ax.set_xlabel("Lag (semanas)")
ax.set_ylabel("")
ax.set_title("Correlação Spearman: Variáveis Climáticas/Satélite × Dengue (2020–2025)")
plt.tight_layout()
fig.savefig(FIG_DIR / "02_heatmap_correlacoes.png")
plt.show()
print("✅ Figura salva: 02_heatmap_correlacoes.png")

## 4. Scatter Plots — Features vs Dengue (lag ótimo)

In [ ]:
cruzado_clima = cruzar_dengue_clima(df_dengue, df_clima, lag=8)
cruzado_sat = cruzar_dengue_clima(df_dengue, df_satelite, lag=8)

df_plot = cruzado_clima[["ano_epi", "semana_epi", "p_inc100k",
                          "temperature_2m_min_lag8"]].merge(
    cruzado_sat[["ano_epi", "semana_epi", "ndvi_lag8", "lst_night_c_lag8"]],
    on=["ano_epi", "semana_epi"], how="inner",
).dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Temp min
axes[0].scatter(df_plot["temperature_2m_min_lag8"], df_plot["p_inc100k"],
                alpha=0.5, c="steelblue", edgecolors="navy", s=30)
axes[0].set_xlabel("Temp. Mínima (°C) — lag 8 sem")
axes[0].set_ylabel("Incidência por 100k hab.")
axes[0].set_title("Temp. Mínima vs Dengue")

# NDVI
axes[1].scatter(df_plot["ndvi_lag8"], df_plot["p_inc100k"],
                alpha=0.5, c="forestgreen", edgecolors="darkgreen", s=30)
axes[1].set_xlabel("NDVI — lag 8 sem")
axes[1].set_ylabel("")
axes[1].set_title("NDVI vs Dengue")

# LST Night
axes[2].scatter(df_plot["lst_night_c_lag8"], df_plot["p_inc100k"],
                alpha=0.5, c="orangered", edgecolors="darkred", s=30)
axes[2].set_xlabel("LST Noturna (°C) — lag 8 sem")
axes[2].set_ylabel("")
axes[2].set_title("LST Noturna vs Dengue")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle("Scatter: Variáveis Climáticas/Satélite × Dengue (lag 8 sem, 2020–2025)", y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / "03_scatter_features_dengue.png")
plt.show()
print("✅ Figura salva: 03_scatter_features_dengue.png")

## 5. Previsão vs Observado — Walk-Forward Validation

In [ ]:
# Carrega previsões do walk-forward
prev = pd.read_csv("data/processed/previsoes_walkforward_2020_2025.csv")

# Cria coluna de data
prev["data"] = prev.apply(
    lambda r: Week(int(r["ano_epi"]), int(r["semana_epi"]), system="iso").startdate(),
    axis=1,
)
prev["data"] = pd.to_datetime(prev["data"])

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(prev["data"], prev["real"], color="black", linewidth=1.5,
        label="Observado", zorder=3)
ax.plot(prev["data"], prev["prev_rf_completo"], color="dodgerblue", linewidth=1.2,
        alpha=0.85, label="RF (clima + autorregressivo)")
ax.plot(prev["data"], prev["prev_baseline"], color="gray", linewidth=1,
        alpha=0.6, linestyle="--", label="Baseline (média móvel 4 sem)")

ax.fill_between(prev["data"], prev["real"], prev["prev_rf_completo"],
                alpha=0.15, color="dodgerblue")

ax.set_xlabel("Data")
ax.set_ylabel("Incidência por 100 mil hab.")
ax.set_title("Previsão vs Observado — Walk-Forward Validation (2021–2025)")
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
fig.savefig(FIG_DIR / "04_previsao_vs_real.png")
plt.show()
print("✅ Figura salva: 04_previsao_vs_real.png")

## 6. Importância das Features (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Prepara dados para treinar modelo final
cruzado_c = cruzar_dengue_clima(df_dengue, df_clima, lag=8)
cruzado_s = cruzar_dengue_clima(df_dengue, df_satelite, lag=8)

df_model = cruzado_c[["ano_epi", "semana_epi", "p_inc100k",
                       "temperature_2m_min_lag8"]].merge(
    cruzado_s[["ano_epi", "semana_epi", "ndvi_lag8"]],
    on=["ano_epi", "semana_epi"], how="inner",
)
df_model = df_model.sort_values(["ano_epi", "semana_epi"]).reset_index(drop=True)

# Adiciona autorregressivas
for lag in [1, 2, 3, 4]:
    df_model[f"casos_lag{lag}"] = df_model["p_inc100k"].shift(lag)
df_model = df_model.dropna().reset_index(drop=True)

feature_cols = ["temperature_2m_min_lag8", "ndvi_lag8",
                "casos_lag1", "casos_lag2", "casos_lag3", "casos_lag4"]

rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                           min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(df_model[feature_cols], df_model["p_inc100k"])

# Plot
importancias = pd.DataFrame({
    "feature": feature_cols,
    "importancia": rf.feature_importances_,
}).sort_values("importancia", ascending=True)

# Nomes mais limpos
rename_feat = {
    "temperature_2m_min_lag8": "Temp. Mínima (lag 8)",
    "ndvi_lag8": "NDVI (lag 8)",
    "casos_lag1": "Casos sem. anterior",
    "casos_lag2": "Casos 2 sem. atrás",
    "casos_lag3": "Casos 3 sem. atrás",
    "casos_lag4": "Casos 4 sem. atrás",
}
importancias["feature_label"] = importancias["feature"].map(rename_feat)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2196F3" if "Casos" in f else "#4CAF50" if "NDVI" in f else "#FF9800"
          for f in importancias["feature_label"]]
ax.barh(importancias["feature_label"], importancias["importancia"], color=colors, edgecolor="white")
ax.set_xlabel("Importância (Gini)")
ax.set_title("Importância das Features — Random Forest")

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2196F3", label="Autorregressiva"),
    Patch(facecolor="#4CAF50", label="Satélite"),
    Patch(facecolor="#FF9800", label="Clima ERA5"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
fig.savefig(FIG_DIR / "05_feature_importance.png")
plt.show()
print("✅ Figura salva: 05_feature_importance.png")

## 7. Performance por Ano — RF vs Baseline

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

perf = []
for ano in sorted(prev["ano_epi"].unique()):
    sub = prev[prev["ano_epi"] == ano]
    if len(sub) < 10:
        continue
    perf.append({
        "Ano": ano,
        "MAE RF": mean_absolute_error(sub["real"], sub["prev_rf_completo"]),
        "MAE Baseline": mean_absolute_error(sub["real"], sub["prev_baseline"]),
        "R² RF": r2_score(sub["real"], sub["prev_rf_completo"]),
    })

df_perf = pd.DataFrame(perf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE comparativo
x = np.arange(len(df_perf))
w = 0.35
axes[0].bar(x - w/2, df_perf["MAE Baseline"], w, label="Baseline", color="gray", alpha=0.7)
axes[0].bar(x + w/2, df_perf["MAE RF"], w, label="Random Forest", color="dodgerblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_perf["Ano"].astype(int))
axes[0].set_ylabel("MAE (incidência/100k)")
axes[0].set_title("Erro Absoluto Médio por Ano")
axes[0].legend()

# R² por ano
axes[1].bar(df_perf["Ano"].astype(int), df_perf["R² RF"], color="dodgerblue")
axes[1].axhline(y=0.885, color="red", linestyle="--", alpha=0.5, label="R² global (0.885)")
axes[1].set_ylabel("R²")
axes[1].set_title("R² do Random Forest por Ano")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.suptitle("Performance Comparativa — Walk-Forward Validation (2021–2025)", y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / "06_performance_por_ano.png")
plt.show()
print("✅ Figura salva: 06_performance_por_ano.png")

## 8. Correlação Parcial — Independência das Features

In [ ]:
# Dados da validação
parcial = pd.read_csv("data/processed/correlacao_parcial_2020_2025.csv")

fig, ax = plt.subplots(figsize=(10, 5))

# Cores por significância
colors = ["#4CAF50" if (abs(r) > 0.15 and p < 0.05) else "#BDBDBD"
          for r, p in zip(parcial["r_parcial"], parcial["p_valor"])]

labels = [f"{row['variavel']}\n(ctrl: {row['controlando']})"
          for _, row in parcial.iterrows()]

bars = ax.barh(labels, parcial["r_parcial"], color=colors, edgecolor="white")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.axvline(x=0.15, color="red", linewidth=0.8, linestyle="--", alpha=0.5, label="Limiar (r=0.15)")
ax.axvline(x=-0.15, color="red", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Correlação Parcial (Pearson)")
ax.set_title("Correlação Parcial com Dengue — Controlando Outras Variáveis")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4CAF50", label="Significativa (|r|>0.15, p<0.05)"),
    Patch(facecolor="#BDBDBD", label="Não significativa"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
fig.savefig(FIG_DIR / "07_correlacao_parcial.png")
plt.show()
print("✅ Figura salva: 07_correlacao_parcial.png")

## 9. Tabela Resumo — Resultados Principais

| Métrica | Baseline (MM 4sem) | RF Clima+Autoregr |
|---|---|---|
| MAE | 26.5 | **19.2** |
| RMSE | 46.1 | **38.6** |
| R² | 0.835 | **0.885** |
| MAPE | 37.1% | **25.6%** |

**Ganho do RF sobre baseline: 27.5% em MAE**

### Features selecionadas (com validação)
| Feature | Tipo | Lag | Contribuição |
|---|---|---|---|
| casos_lag1 | Autorregressiva | 1 sem | Importância: 0.976 |
| temperature_2m_min | Clima ERA5 | 8 sem | Parcial: r=0.155, p=0.008 |
| ndvi | Satélite MODIS | 8 sem | Parcial: r=0.327, p<0.001 |
| lst_night_c | Satélite MODIS | 8 sem | ❌ Descartada (VIF=7.8, redundante com temp_min) |


In [ ]:
# Lista todas as figuras geradas
print("\n" + "=" * 60)
print("FIGURAS GERADAS (300 DPI, prontas para publicação)")
print("=" * 60)
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  📊 {f}")
print(f"\nTotal: {len(list(FIG_DIR.glob('*.png')))} figuras")